# Generate Splits for BioBERT 

In [1]:
import sys, os, json, random
from pathlib import Path 

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:

# Correct - ensure data/biobert_splits under PROJECT_ROOT/v03/data/
SPLIT_DIR = PROJECT_ROOT / "v03" / "data" / "biobert_splits"
os.makedirs(SPLIT_DIR, exist_ok=True)
INPUT_FILE = PROJECT_ROOT / "v03" / "data" / "data_wordpiece_tokenized_biobert.jsonl"
TRAIN_FILE = SPLIT_DIR / "train.jsonl"
VAL_FILE = SPLIT_DIR / "val.jsonl"
TEST_FILE = SPLIT_DIR / "test.jsonl"

with open(INPUT_FILE, "r") as f:
    data = [json.loads(line) for line in f]

# Shuffle
random.shuffle(data)

# biobert_splits
n = len(data)
train_end = int(0.8*n)  # 80% train data 
val_end = int(0.9*n)    # 10% val data
train_data = data[:train_end]
val_data   = data[train_end:val_end]
test_data  = data[val_end:] 

# Save helper
def save_jsonl(path, dataset):
    with open(path, "w") as f:
        for row in dataset:
            f.write(json.dumps(row) + "\n")

# Save splits
save_jsonl(TRAIN_FILE, train_data)
save_jsonl(VAL_FILE, val_data)
save_jsonl(TEST_FILE, test_data)

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")


Train: 85013
Val:   10627
Test:  10627


# Upload Datasets to Huggingface

In [2]:
from datasets import load_dataset

data_files = {
"train": f"{TRAIN_FILE}",
"validation": f"{VAL_FILE}",
"test" : f"{TEST_FILE}"
}

dataset = load_dataset("json", data_files=data_files)
print(dataset)

/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 85013 examples [00:00, 1006081.47 examples/s]
Generating validation split: 10627 examples [00:00, 887198.82 examples/s]
Generating test split: 10627 examples [00:00, 775126.40 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels'],
        num_rows: 85013
    })
    validation: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels'],
        num_rows: 10627
    })
    test: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels'],
        num_rows: 10627
    })
})


In [3]:
dataset.push_to_hub("Rogarcia18/symptoms_ner_v03_biobert", private=False)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  6.59ba/s]
Processing Files (1 / 1): 100%|██████████| 5.83MB / 5.83MB, 51.4kB/s  
New Data Upload: 100%|██████████| 5.83MB / 5.83MB, 51.4kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 29.47ba/s]
Processing Files (1 / 1): 100%|██████████|  776kB /  776kB, 88.2kB/s  
New Data Upload: 100%|██████████|  776kB /  776kB, 88.2kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 47.46ba/s]
Processing Files (1 / 1): 100%|██████████|  771kB /  771kB, 24.1kB/s  
New Data Upload: 100%|██████████|  771kB /  771kB, 24.1kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:14<00:00, 14.17s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/Rogarcia18/symptoms_ner_v03_biobert/commit/65b4b43547b26bbd44dc98d1de6a322afc0e7c97', commit_message='Upload dataset', commit_description='', oid='65b4b43547b26bbd44dc98d1de6a322afc0e7c97', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Rogarcia18/symptoms_ner_v03_biobert', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Rogarcia18/symptoms_ner_v03_biobert'), pr_revision=None, pr_num=None)

In [3]:

from huggingface_hub import HfApi, upload_file, hf_hub_download

# Load the label mappings
with open(f"{PROJECT_ROOT}/v03/data/id2label.json", "r") as f:
    id2label = json.load(f)
with open(f"{PROJECT_ROOT}/v03/data/label2id.json", "r") as f:
    label2id = json.load(f)

# Initialize the Hugging Face API
api = HfApi()
repo_id = "Rogarcia18/symptoms_ner_v03_biobert"
# Check existing files in the repository (for information only)
existing_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
print(f"Existing files in repository: {existing_files}")

ConnectionError: (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /api/datasets/Rogarcia18/symptoms_ner_v03_biobert/tree/main?recursive=True&expand=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10d99b610>: Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 57808684-e261-4f1f-8240-b0a5a4221f59)')

In [ ]:
id2label,label2id

In [ ]:
# Upload id2label.json
print("Uploading id2label.json...")
upload_file(
    path_or_fileobj=f"{PROJECT_ROOT}/v03/data/id2label.json",
    path_in_repo="id2label.json",
    repo_id=repo_id,
    repo_type="dataset",
)
print("✓ id2label.json uploaded successfully")

# Upload label2id.json
print("Uploading label2id.json...")
upload_file(
    path_or_fileobj=f"{PROJECT_ROOT}/v03/data/label2id.json",
    path_in_repo="label2id.json",
    repo_id=repo_id,
    repo_type="dataset",
)
print("✓ label2id.json uploaded successfully")

**Test downloading the data**

In [ ]:
# Download and load id2label.json from the hub
id2label_path = hf_hub_download(
    repo_id=repo_id,
    filename="id2label.json",
    repo_type="dataset"
)
with open(id2label_path, "r") as f:
    id2label = json.load(f)
print(f"✓ Loaded id2label.json from hub")
print(f"  Total labels: {len(id2label)}")
print(f"  First 5 labels: {dict(list(id2label.items())[:5])}")

# Download and load label2id.json from the hub
label2id_path = hf_hub_download(
    repo_id=repo_id,
    filename="label2id.json",
    repo_type="dataset"
)
with open(label2id_path, "r") as f:
    label2id = json.load(f)
print(f"\n✓ Loaded label2id.json from hub")
print(f"  Total labels: {len(label2id)}")
print(f"  First 5 labels: {dict(list(label2id.items())[:5])}")

# Now you can use id2label and label2id in your code!